# baia-vision-poc — demo no Google Colab

Sobe o **front de teste** da POC com um **link público temporário** (via cloudflared, sem conta).
Roda com a CPU do Colab — e fica bem mais rápido se você ligar a GPU grátis.

**Como usar:**
1. (Opcional, recomendado) `Ambiente de execução` → `Alterar o tipo de ambiente` → **T4 GPU**.
2. Rode as células em ordem (▶️).
3. Na última célula, clique no link `https://....trycloudflare.com` e suba um vídeo.

> ⚠️ Use somente **footage royalty-free** (Pexels/Pixabay/Mixkit). Esta POC não é sistema de segurança de vida, não identifica pessoas e não lê placas.

In [ ]:
# 1) Baixa o código
![ -d ReconhecimentoPy ] || git clone -b main https://github.com/AliceCullen-html/ReconhecimentoPy.git
%cd /content/ReconhecimentoPy

In [ ]:
# 2) Instala dependências + cloudflared (túnel público)
#    O Colab já traz torch/opencv/numpy; instalamos o resto.
!pip install -q ultralytics flask gunicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print('✅ dependências prontas')

In [ ]:
# 3) Sobe o app e abre o link público
import os, subprocess, time, re, threading

# Colab tem CPU/GPU decente: qualidade quase cheia. Ajuste se quiser.
os.environ.update({
    'MAX_UPLOAD_MB': '300',
    'PROC_MAX_WIDTH': '1280',   # 720p passa inteiro; limita só 4K
    'PROC_FRAME_STRIDE': '1',   # detecta todos os frames
})

def _drain(proc):
    """Esvazia a saída do processo para não travar o buffer."""
    for _ in proc.stdout:
        pass

# Servidor (gunicorn com threads: atende o polling de status enquanto processa)
server = subprocess.Popen(
    ['gunicorn', 'webapp.app:app', '--bind', '127.0.0.1:7860',
     '--workers', '1', '--worker-class', 'gthread', '--threads', '4',
     '--timeout', '1200'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
threading.Thread(target=_drain, args=(server,), daemon=True).start()
print('⏳ subindo o servidor (carrega o modelo)…')
time.sleep(10)

# Túnel público (cloudflared quick tunnel — sem conta/token)
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url, start = None, time.time()
for line in tunnel.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    if time.time() - start > 90:
        break
threading.Thread(target=_drain, args=(tunnel,), daemon=True).start()

print('\n' + '=' * 64)
if url:
    print('🚀 APP PÚBLICO:  ' + url)
else:
    print('Não consegui capturar a URL — rode esta célula de novo.')
print('=' * 64)
print('Deixe esta célula rodando. O link morre quando o notebook fecha.')

**Para parar:** `Ambiente de execução` → `Interromper execução`.

**Dica de velocidade:** com a GPU T4 ligada, a inferência YOLO fica muito mais rápida.
Para vídeos maiores em CPU, aumente `PROC_FRAME_STRIDE` (ex.: `2`) na célula 3.